In [ ]:
!git clone https://github.com/JLansey/EgyptGPT.git

Cloning into 'EgyptGPT'...
remote: Enumerating objects: 726, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 726 (delta 18), reused 35 (delta 17), pack-reused 682 (from 1)
Receiving objects: 100% (726/726), 963.17 KiB | 2.12 MiB/s, done.
Resolving deltas: 100% (403/403), done.


In [ ]:
cd EgyptGPT

/content/EgyptGPT


In [ ]:
!bash setup.sh

Submodule 'hiero_transformer' (https://github.com/mattia-decao/hiero-transformer.git) registered for path 'hiero_transformer'
Cloning into '/content/EgyptGPT/hiero_transformer'...
Submodule path 'hiero_transformer': checked out '86a7bd77b03fec99fc01d76b2c24291c474c8694'
Archive:  hiero_transformer/training_data.zip
  inflating: data/egypt_char/training_data.json  
Archive:  hiero_transformer/test_and_validation_data.zip
  inflating: data/egypt_char/test_data.json  
  inflating: data/egypt_char/validation_data.json  
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 9.5 MB/s eta 0:00:

In [ ]:
!python init_model.py

2024-10-25 19:25:29.109856: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-25 19:25:29.126002: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-25 19:25:29.148461: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-25 19:25:29.158420: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-25 19:25:29.174694: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
# prompt: autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
import matviz
from matviz.etl import write_string
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

from glyph_utils import translate_long_text, gardiner_to_hieroglyphics


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# ETL the glyph data
from data.egypt_char import prepare

input path is: /content/EgyptGPT/data/egypt_char/cleaned_graphics.txt
/content/EgyptGPT/data/egypt_char/validation_data.json
CLEANING THE DATA
Ci sono 6 files.
Caricati 61605 datapoints.


100%|██████████| 61605/61605 [00:00<00:00, 80752.73it/s]



CLEANING COMPLETE.
length of dataset in characters: 1,464,710
all the unique characters: 
 ./0123456789?ABCDEFGHIJKLMNOPQRSTUVWXYZafh
vocab size: 44
train has 1,318,239 tokens
val has 146,471 tokens


In [ ]:
cleaned_graphics = matviz.etl.read_string('data/egypt_char/cleaned_graphics.txt')
cleaned_graphics = cleaned_graphics.split('\n')


In [ ]:
len(cleaned_graphics)


18870

In [ ]:
len("\n".join(cleaned_graphics))

1464710

In [ ]:
cleaned_graphics[0]

''

In [ ]:
from matviz.helpers_graphing import *

In [ ]:

# tic()
# for ii in range(10):
#     hieroglyphic_text = cleaned_graphics[ii]
#     # print(f"  raw:{hieroglyphic_text}")
#     # print(f"glyph:{gardiner_to_hieroglyphics(hieroglyphic_text)}")
#     translation = translate_long_text(hieroglyphic_text, max_chars=64)
#     toc()
#     # print(f"engli: {translation}")
#     # print()


In [ ]:
# took 16.5 seconds to do 10

In [ ]:
import wandb
wandb.login()

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
!python train.py config/train_egypt_char.py

Overriding config with config/train_egypt_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-glyphs-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = True # override via command line if you like
wandb_project = 'glyph-char'
wandb_run_name = 'mini-gpt'

dataset = 'egypt_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 746 bytes | 746.00 KiB/s, done.
From https://github.com/JLansey/EgyptGPT
   ee5a4a5..8e5a63e  master     -> origin/master
Updating ee5a4a5..8e5a63e
Fast-forward
 sample.py | 39 ++++++++++++++++++++++++++++++++-------
 1 file changed, 32 insertions(+), 7 deletions(-)


In [ ]:
mkdir out

In [ ]:
!python sample.py --out_dir=out-glyphs-char


Overriding: out_dir = out-glyphs-char
/content/EgyptGPT/sample.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)
n

In [ ]:
!mkdir out

In [ ]:
!python sample.py --out_dir=out-glpyhs-char

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Then save directly to a path like:
# df.to_csv('/content/drive/MyDrive/your_file.csv')

Mounted at /content/drive


In [ ]:
from tqdm import tqdm

In [ ]:
from google.colab import files

In [ ]:

save_path = f'/content/drive/MyDrive/'  # or you can name the actual folder path if you know it


In [ ]:
to_print.to_csv(f'{save_path}out_translated_{temperature}.csv')

In [ ]:
for temperature in [0.7, 1.0]:

    gen_data = read_string(f"out/output_{temperature}.txt")
    samples_all = gen_data.split("\n")
    samples_all = [w for w in samples_all if w]

    all_translations = []
    all_glyphs = []
    for idx, cur in tqdm(enumerate(samples_all)):
        translation = translate_long_text(cur)
        glyph = gardiner_to_hieroglyphics(cur)

        all_translations.append(translation)
        all_glyphs.append(glyph)
        if idx % 500 == 0:
            print(f"{idx}: {cur} \n{idx}: {translation}")

        if idx % 100 == 0:
            print(f"SAVING TO FILE {idx}: {cur} \n{idx}: {translation}")

            filepath = f"out/translated_{temperature}.csv"
            to_print = np.concatenate([np.array([samples_all[:len(all_glyphs)]]), np.array([all_glyphs]), np.array([all_translations])]).T
            to_print = pd.DataFrame(to_print, columns=['gardiner', 'glyph', 'english'])
            to_print.to_csv(filepath)
            to_print.to_csv(f'{save_path}out_translated_{temperature}.csv')


    to_print = np.concatenate([np.array([samples_all]), np.array([all_glyphs]), np.array([all_translations])]).T
    to_print = pd.DataFrame(to_print, columns=['gardiner', 'glyph', 'english'])
    to_print.to_csv(filepath)
    files.download(filepath)



0it [00:00, ?it/s]

0: G39 X1 I9 M23 X1 D21 Aa1 V28 M2 N35 W24 X1 G43 
0: His daughter, the manager of the royal property Henut.
SAVING TO FILE 0: G39 X1 I9 M23 X1 D21 Aa1 V28 M2 N35 W24 X1 G43 
0: His daughter, the manager of the royal property Henut.


101it [01:06,  1.77it/s]

SAVING TO FILE 100: M17 G43 D21 D36 D21 D21 D21 X1 X1 S2 X1 N35 M17 X1 I9 A1 F31 S29 X1 B3 U6 M17 M17 I9 G17 R4 X1 Q3  
100: His father was born, his beloved in peace.


201it [02:04,  1.18it/s]

...

9000: D37 O34 A1 Z1 Z1 G28 G17 V28 D6 D2 Z1 N29 G1 D58 X1 F51 I9 D4 N35 S29 N35 Z2 G41 G1 D36 A1 U28 G1 M17 Z7 H8 G17 F32 Q3 Z7 X1 F51 Z3 D35 N35 E34 N35 N35 I9 G17 F26 N35 W24 Z7 O1 S29 F35 I9 D21 
9000: A man was given to his chest after they had done it. A puppy in the abdomen: it does not be in the inside of her good.
SAVING TO FILE 9000: D37 O34 A1 Z1 Z1 G28 G17 V28 D6 D2 Z1 N29 G1 D58 X1 F51 I9 D4 N35 S29 N35 Z2 G41 G1 D36 A1 U28 G1 M17 Z7 H8 G17 F32 Q3 Z7 X1 F51 Z3 D35 N35 E34 N35 N35 I9 G17 F26 N35 W24 Z7 O1 S29 F35 I9 D21 
9000: A man was given to his chest after they had done it. A puppy in the abdomen: it does not be in the inside of her good.


9101it [1:15:47,  2.00it/s]

SAVING TO FILE 9100: Aa1 S29 U35 A24 D4 D21 X1 Z2 D21 D21 S29 X1 D21 Y5  
9100: Remove what is done against it.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
[len(samples_all), len(all_glyphs),len(all_translations)]

[8408, 1155, 1155]

In [ ]:
to_print = np.concatenate([np.array([samples_all[:1155]]), np.array([all_glyphs]), np.array([all_translations])]).T
to_print = pd.DataFrame(to_print, columns=['gardiner', 'glyph', 'english'])
to_print.to_csv(f"out/translated_{temperature}.csv")


In [ ]:
write_csv("translated_samples.csv", to_print, names=['a', 'b', 'c'])


TypeError: write_csv() got an unexpected keyword argument 'names'

In [ ]:

write_csv("translated_samples.csv", np.concatenate([np.array([samples_all]), np.array([all_translations])]).T)


True

In [ ]:
# all_translations

Overriding: out_dir = out-egypt-char
/content/EgyptGPT/sample.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)
nu